# MEMOIR Experiments — Session 2: Ablations + Sensitivity

## Before starting
1. Run Session 1 (`memoir_kaggle_session1.ipynb`) and **Save Version**
2. **Add Data** → add Session 1 output as `memoir-session1-output`
3. **Add Data** → attach `memoir-amazon-processed` (same as Session 1)
4. Accelerator → **GPU T4 x2**, Internet → **On**
5. Set `GITHUB_REPO` and verify `SESSION1_OUTPUT` path in Cell 2

## What this session runs
| Step | Contents | Est. time |
|------|----------|-----------|
| Ablations | w/o Evolution CL, w/o Direction Loss, w/o Temporal Seg., w/ Random Items | ~4 h |
| Sensitivity | temperature (0.03, 0.2), windows (3, 12) | ~4 h |

In [ ]:
# Cell 1: Check GPU
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 2: Configuration
import os

GITHUB_REPO     = 'https://github.com/louiezzang/MEMOIR.git'
DATASET_NAME    = 'datasets'  # adjust if your Kaggle dataset mount name differs
SESSION1_OUTPUT = '/kaggle/input/memoir-session1-output'  # Session 1 saved output

REPO_DIR   = '/kaggle/working/MEMOIR'
DATA_INPUT = f'/kaggle/input/{DATASET_NAME}'
WORK_DIR   = '/kaggle/working'
CKPT_DIR   = f'{WORK_DIR}/checkpoints'
LOG_DIR    = f'{WORK_DIR}/logs'

assert GITHUB_REPO, 'Set GITHUB_REPO'
if not os.path.exists(DATA_INPUT):
    for name in os.listdir('/kaggle/input'):
        candidate = f'/kaggle/input/{name}'
        for root, dirs, files in os.walk(candidate):
            if any(f.endswith('.parquet') for f in files):
                DATASET_NAME = name
                DATA_INPUT = candidate
                break
        if os.path.exists(DATA_INPUT):
            break
assert os.path.exists(DATA_INPUT), f'Attach dataset. Check /kaggle/input/ contents: {os.listdir("/kaggle/input/")}'
assert os.path.exists(SESSION1_OUTPUT), f'Attach Session 1 output. Check /kaggle/input/ contents: {os.listdir("/kaggle/input/")}'
print(f'Dataset: {DATASET_NAME}')
print(f'Dataset path: {DATA_INPUT}')
print(f'Session 1 output: {SESSION1_OUTPUT}')

In [ ]:
# Cell 3: Clone / pull repo
import os
if not os.path.exists(REPO_DIR):
    os.system(f'git clone {GITHUB_REPO} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull')
os.chdir(REPO_DIR)
os.system(f'find {REPO_DIR} -name "__pycache__" -exec rm -rf {{}} + 2>/dev/null; true')
print(f'Working dir: {os.getcwd()}')

In [ ]:
# Cell 4: Install dependencies (preserve Kaggle's CUDA-enabled PyTorch)
import torch, subprocess
_torch_version = torch.__version__

!pip install -q \
    transformers>=4.40.0 \
    peft>=0.10.0 \
    sentence-transformers>=3.0.0 \
    datasets>=2.19.0 \
    bitsandbytes \
    pandas pyarrow scikit-learn scipy pyyaml tqdm

# Check if pip replaced CUDA PyTorch (use subprocess to avoid reload issues)
check = subprocess.run(
    ['python', '-c', 'import torch; print(torch.cuda.is_available()); print(torch.__version__)'],
    capture_output=True, text=True
)
lines = check.stdout.strip().split('\n')
cuda_ok = lines[0] == 'True' if lines else False
new_version = lines[1] if len(lines) > 1 else 'unknown'

if not cuda_ok:
    print(f'WARNING: pip replaced CUDA PyTorch ({_torch_version}) with CPU build ({new_version})!')
    print('Reinstalling CUDA PyTorch...')
    subprocess.run(['pip', 'install', '-q', f'torch=={_torch_version}',
                    '--index-url', 'https://download.pytorch.org/whl/cu121'], check=True)
    check2 = subprocess.run(
        ['python', '-c', 'import torch; print(torch.cuda.is_available())'],
        capture_output=True, text=True
    )
    assert 'True' in check2.stdout, 'Failed to restore CUDA PyTorch!'
    print(f'Restored PyTorch {_torch_version} with CUDA.')
else:
    print(f'Done. PyTorch {new_version}, CUDA=True')

In [ ]:
# Cell 5: Link data & directories + import Session 1 outputs
import os, shutil

LOCAL_PROCESSED = f'{REPO_DIR}/data/processed/amazon'

# Find the directory containing the parquet files
data_source = None
for root, dirs, files in os.walk(DATA_INPUT):
    if any(f.endswith('.parquet') for f in files):
        data_source = root
        break

assert data_source, f'No parquet files found under {DATA_INPUT}'

# Ensure parent directory exists (data/processed/ is gitignored)
os.makedirs(os.path.dirname(LOCAL_PROCESSED), exist_ok=True)

if os.path.islink(LOCAL_PROCESSED):
    os.unlink(LOCAL_PROCESSED)
if os.path.exists(LOCAL_PROCESSED):
    shutil.rmtree(LOCAL_PROCESSED)

shutil.copytree(data_source, LOCAL_PROCESSED)
print(f'Data source: {data_source}')
print(f'Copied files: {os.listdir(LOCAL_PROCESSED)}')

# Verify critical files exist
for required in ['train_samples.parquet', 'val_samples.parquet', 'test_samples.parquet']:
    assert os.path.exists(f'{LOCAL_PROCESSED}/{required}'), f'Missing {required} after copy!'
print('All required parquet files verified.')

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
local_ckpt = f'{REPO_DIR}/checkpoints'
local_log  = f'{REPO_DIR}/logs'
if not os.path.exists(local_ckpt):
    os.symlink(CKPT_DIR, local_ckpt)
if not os.path.exists(local_log):
    os.symlink(LOG_DIR, local_log)

# Import Session 1 logs and checkpoints
for subdir in ['logs', 'checkpoints']:
    src = f'{SESSION1_OUTPUT}/{subdir}'
    if os.path.exists(src):
        dst = CKPT_DIR if subdir == 'checkpoints' else LOG_DIR
        for item in os.listdir(src):
            item_src = f'{src}/{item}'
            item_dst = f'{dst}/{item}'
            if not os.path.exists(item_dst):
                if os.path.isdir(item_src):
                    shutil.copytree(item_src, item_dst)
                else:
                    shutil.copy2(item_src, item_dst)
        print(f'Imported {subdir} from Session 1: {os.listdir(src)}')
    else:
        print(f'WARNING: {src} not found in Session 1 output')

print(f'\nAll logs: {os.listdir(LOG_DIR)}')

In [ ]:
# Cell 6: Write Kaggle config (GPU-optimised)
import yaml, os

with open(f'{REPO_DIR}/configs/default.yaml') as f:
    config = yaml.safe_load(f)

config['data']['processed_dir']                  = f'{REPO_DIR}/data/processed'
config['model']['llm_load_in_4bit']              = False
config['model']['freeze_llm']                    = True
config['training']['batch_size']                 = 128
config['training']['gradient_accumulation_steps'] = 1
config['training']['fp16']                       = True
config['logging']['save_dir']                    = CKPT_DIR
config['logging']['log_dir']                     = LOG_DIR
config['logging']['save_every']                  = 5

KAGGLE_CONFIG = f'{REPO_DIR}/configs/kaggle.yaml'
with open(KAGGLE_CONFIG, 'w') as f:
    yaml.dump(config, f)

print('configs/kaggle.yaml written')
print(f"  processed_dir={config['data']['processed_dir']}")
print(f"  batch_size={config['training']['batch_size']}, fp16={config['training']['fp16']}")

---
## Ablation Study

In [ ]:
# Helper: run a MEMOIR variant
import sys, subprocess, json, os
sys.path.insert(0, REPO_DIR)

SUBPROCESS_ENV = {**os.environ, 'PYTHONPATH': REPO_DIR, 'PYTHONDONTWRITEBYTECODE': '1'}

def run_memoir_variant(label, extra_args, log_suffix):
    log_dir = f'{LOG_DIR}/memoir_amazon{log_suffix}'
    metrics_path = f'{log_dir}/metrics.json'

    if os.path.exists(metrics_path):
        data = json.loads(open(metrics_path).read())
        evals = [x for x in data if x.get('type') == 'eval']
        if evals:
            best = max(evals, key=lambda x: x.get('ndcg@10', 0))
            print(f'[SKIP] {label} '
                  f'(HR@10={best.get("hr@10",0):.4f}, NDCG@10={best.get("ndcg@10",0):.4f})')
            return

    print(f'[RUN ] {label}...')
    subprocess.run(
        ['python', 'train.py', '--config', KAGGLE_CONFIG, '--dataset', 'amazon',
         '--log-suffix', log_suffix] + extra_args,
        check=True,
        env=SUBPROCESS_ENV
    )
    if os.path.exists(metrics_path):
        data = json.loads(open(metrics_path).read())
        evals = [x for x in data if x.get('type') == 'eval']
        if evals:
            best = max(evals, key=lambda x: x.get('ndcg@10', 0))
            print(f'[DONE] {label} '
                  f'HR@10={best.get("hr@10",0):.4f}, NDCG@10={best.get("ndcg@10",0):.4f}')

print('Helper ready.')

In [ ]:
# Ablation study (~4 h)
run_memoir_variant('w/o Evolution CL',  ['--no-evo-cl'],   '_no_evo_cl')
run_memoir_variant('w/o Direction Loss',['--no-dir-loss'],  '_no_dir_loss')
run_memoir_variant('w/o Temporal Seg.', ['--no-temporal'],  '_no_temporal')
run_memoir_variant('w/ Random Items',   ['--override', 'model.item_encoder.type=random'], '_random_items')

---
## Hyperparameter Sensitivity

In [ ]:
# Sensitivity: temperature and window count (~4 h)
# Default values (tau=0.07, W=6) reuse MEMOIR main from Session 1
run_memoir_variant('tau=0.03', ['--override', 'model.temperature=0.03'], '_tau0.03')
run_memoir_variant('tau=0.20', ['--override', 'model.temperature=0.2'],  '_tau0.2')

run_memoir_variant('W=3',  ['--override', 'model.num_memory_windows=3'],  '_W3')
run_memoir_variant('W=12', ['--override', 'model.num_memory_windows=12'], '_W12')

---
## Full Results Summary

Combines Session 1 (baselines + MEMOIR main) and Session 2 (ablations + sensitivity).

In [ ]:
# Full results summary
import json, os

def best_metrics(log_dir):
    p = f'{log_dir}/metrics.json'
    if not os.path.exists(p):
        return None
    data = json.loads(open(p).read())
    evals = [x for x in data if x.get('type') == 'eval']
    return max(evals, key=lambda x: x.get('ndcg@10', 0)) if evals else None

fmt = lambda v: f'{v:.4f}' if v is not None else '  --  '

# Main results (from Session 1)
print('=== Main Results ===')
for name, key in [('GRU4Rec','gru4rec'), ('SASRec','sasrec'), ('CL4SRec','cl4srec'),
                  ('DuoRec','duorec'), ('SRA-CL','sracl'),
                  ('SASRec-Text','sasrec_text'), ('UniSRec','unisrec'),
                  ('MEMOIR','memoir')]:
    m = best_metrics(f'{LOG_DIR}/{key}_amazon')
    if m:
        print(f"{name:<22} HR@10={fmt(m.get('hr@10'))} NDCG@10={fmt(m.get('ndcg@10'))} "
              f"HR@20={fmt(m.get('hr@20'))} MRR={fmt(m.get('mrr'))}")
    else:
        print(f'{name:<22} (not run)')

# Ablations (from Session 2)
print('\n=== Ablation Study ===')
for name, suffix in [('MEMOIR Full',''), ('w/o Evo CL','_no_evo_cl'),
                     ('w/o Dir Loss','_no_dir_loss'), ('w/o Temporal','_no_temporal'),
                     ('w/ Random Items','_random_items')]:
    m = best_metrics(f'{LOG_DIR}/memoir_amazon{suffix}')
    if m:
        print(f"{name:<22} HR@10={fmt(m.get('hr@10'))} NDCG@10={fmt(m.get('ndcg@10'))}")
    else:
        print(f'{name:<22} (not run)')

# Sensitivity (from Session 2)
print('\n=== Sensitivity (HR@10) ===')
for name, suffix in [('tau=0.03','_tau0.03'), ('tau=0.07 (default)',''), ('tau=0.20','_tau0.2'),
                     ('W=3','_W3'), ('W=6 (default)',''), ('W=12','_W12')]:
    m = best_metrics(f'{LOG_DIR}/memoir_amazon{suffix}')
    hr = fmt(m.get('hr@10') if m else None)
    print(f'  {name:<22} HR@10={hr}')

print('\nSave Version to persist all results!')

---
## After this session
Click **Save Version** (top right) → *Save & Run All* to persist all results.